In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.preprocessing import MinMaxScaler
from mlxtend.feature_selection import SequentialFeatureSelector 
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline, make_pipeline

import warnings
warnings.filterwarnings('ignore')

**The data provided are responses for a survey conducted in 2015 on a sample of 70 thousand people to assess the factors leading to getting diagnosed with diabetes.**

In [2]:
df = pd.read_csv("diabetes.csv")
df.head()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,3.0,5.0,30.0,0.0,1.0,4.0,6.0,8.0
1,0.0,1.0,1.0,1.0,26.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,12.0,6.0,8.0
2,0.0,0.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,10.0,0.0,1.0,13.0,6.0,8.0
3,0.0,1.0,1.0,1.0,28.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,0.0,3.0,0.0,1.0,11.0,6.0,8.0
4,0.0,0.0,0.0,1.0,29.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,8.0,5.0,8.0


In [3]:
df = df.sample(12000, random_state=49)


### Exploratory Data Analysis ###

In [4]:
df.shape

(12000, 22)

In [5]:
df.isna().sum()

Diabetes_binary         0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

In [6]:
df.columns

Index(['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
       'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education',
       'Income'],
      dtype='object')

In [7]:
for col in df.columns:
    print(f'{col}:', df[col].unique())

Diabetes_binary: [0. 1.]
HighBP: [0. 1.]
HighChol: [0. 1.]
CholCheck: [1. 0.]
BMI: [24. 28. 22. 42. 31. 49. 26. 33. 34. 30. 32. 25. 21. 29. 19. 39. 43. 37.
 47. 23. 36. 35. 41. 27. 40. 55. 20. 38. 44. 48. 45. 18. 46. 50. 87. 51.
 67. 15. 58. 73. 53. 54. 63. 75. 17. 61. 52. 16. 59. 77. 79. 57. 13. 82.
 72. 66. 60. 92. 84. 65. 64. 62. 56. 89. 81. 14. 71.]
Smoker: [0. 1.]
Stroke: [0. 1.]
HeartDiseaseorAttack: [0. 1.]
PhysActivity: [1. 0.]
Fruits: [1. 0.]
Veggies: [1. 0.]
HvyAlcoholConsump: [0. 1.]
AnyHealthcare: [1. 0.]
NoDocbcCost: [1. 0.]
GenHlth: [2. 4. 5. 3. 1.]
MentHlth: [ 0. 30.  1.  8.  5. 10. 15. 20.  2. 25. 12.  4. 29.  3.  7.  6. 14. 21.
 18. 23. 13. 17. 26. 19. 28. 27.  9. 16. 11. 22. 24.]
PhysHlth: [ 2. 10. 30. 14.  0.  3.  1.  5. 15.  4.  8. 20.  7. 18. 29.  6.  9. 28.
 25. 12. 16. 27. 21. 13. 17. 24. 11. 22. 23. 26. 19.]
DiffWalk: [0. 1.]
Sex: [0. 1.]
Age: [ 6. 10. 13.  8.  9. 12. 11.  7.  4.  1.  3.  5.  2.]
Education: [6. 4. 3. 5. 2. 1.]
Income: [5. 1. 4. 7. 6. 8. 3. 2.]


In [8]:
fig = px.histogram(df, x='BMI', title= 'BMI Distribution')
fig.update_layout(template='plotly_white', yaxis_title=None)
fig.show()

In [9]:
fig = px.box(df, x='BMI', title= 'BMI Distribution')
fig.update_layout(template='plotly_white', yaxis_title=None)
fig.show()

In [10]:
fig = px.box(df, x='MentHlth', title= 'How many days in the previous month your mental health was bad', labels={'MentHlth': 'Mental Health'})
fig.update_layout(template='plotly_white', yaxis_title=None)
fig.show()

In [11]:
fig = px.box(df, x='PhysHlth', title= 'How many days in the previous month your physical health was bad', labels={'PhysHlth': 'Physical Health'})
fig.update_layout(template='plotly_white', yaxis_title=None)
fig.show()

In [12]:
df_c = df.copy()
df_c['Sex'] = df_c['Sex'].map({0: 'Female', 1: 'Male'})
bin_cols = ['Diabetes_binary', 'HighBP', 'HighChol', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity',
       'HvyAlcoholConsump']

for col in bin_cols:
    df_c[col] = df_c[col].map({0: 'No', 1:'Yes'})
bin_cols = bin_cols + ['Sex']

In [13]:
titles = ['Diabetes', 'High blood pressure', 'High cholestrol', 'Smoker', 'Stroke', 'Heart Disease/Attack', 'Physical Activity',
          'Alcholicism', 'Sex']
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=titles,
    specs=[[{'type': 'domain'}]*3]*3  # each cell holds a pie chart
    )

for i, (col, title) in enumerate(zip(bin_cols, titles)):
    counts = df_c[col].value_counts().reset_index()
    counts.columns = [col, 'Count']

    row = i // 3 + 1
    col_pos = i % 3 + 1

    fig.add_trace(
        go.Pie(labels=counts[col], values=counts['Count'], textinfo='percent+label'),
        row=row, col=col_pos
    )

# Layout customization
fig.update_layout(
    height=1100,
    title_text='Distribution of Binary Columns',
    template='plotly_white',
    showlegend=False
)

fig.show()


In [14]:
factors = ['HighBP', 'HighChol', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'HvyAlcoholConsump']
fac_titles = ['High blood pressure', 'High cholestrol', 'Smoker', 'Stroke', 'Heart Disease/Attack', 'Physical Activity', 'Alcholic']

for col, title in zip(bin_cols, fac_titles):
    fig = px.histogram(df_c, x=col, color='Diabetes_binary', labels={f'{col}': f'{title}', 'Diabetes_binary': 'Diabetic'}, title=title)
    fig.update_layout(template='plotly_white', yaxis_title=None, xaxis_title=title)
    fig.show()

In [15]:
df_c['GenHlth'] = df_c['GenHlth'].map({1: 'Excellent', 2: 'Very Good', 3: 'Good', 4: 'Fair', 5: 'Poor'})
health_order = ['Excellent', 'Very Good', 'Good', 'Fair', 'Poor']
df_c['GenHlth'] = pd.Categorical(df_c['GenHlth'], categories=health_order, ordered=True)
counts = df_c['GenHlth'].value_counts().sort_index().reset_index()
counts.columns = ['GenHlth', 'counts']

fig = px.bar(counts, x='counts', y='GenHlth', orientation='h', color='GenHlth', labels={'GenHlth': 'General Health'})
fig.update_layout(template='plotly_white', showlegend=False, xaxis_title=None)
fig.show()

In [16]:
df_c['Age'] = df_c['Age'].map({1: '18-24', 2: '25-29', 3: '30-34', 4: '35-39', 5: '40-44', 6: '45-49', 7: '50-54',8: '55-59', 9: '60-64', 10: '65-69', 11: '70-74', 12: '75-79', 13: '80+'})
order = ['18-24','25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80+']

df_c['Age'] = pd.Categorical(df_c['Age'], categories=order, ordered=True)
counts = df_c['Age'].value_counts().sort_index().reset_index()
counts.columns = ['Age', 'count']

fig = px.bar(counts, x='Age', y='count', title='Age Distribution')
fig.update_layout(template='plotly_white', showlegend=False, yaxis_title=None)
fig.show()

### Dimensionality Reduction ###

Preprocessing

In [17]:
edu = {1: 'Never Attended', 2: 'Elementary', 3: 'High School_dropout', 4: 'High_School_graduate', 5: 'College_dropout', 6: 'College_graduate'}
df['Education'] = df['Education'].map(edu)


In [18]:
# one hot encoding
dummies = pd.get_dummies(df.Education, dtype='int')
df = pd.concat([df, dummies], axis=1)
df = df.drop(['Education', 'Never Attended'], axis=1)

In [19]:
corr_matrix = df.corr()
corr_matrix

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,PhysHlth,DiffWalk,Sex,Age,Income,College_dropout,College_graduate,Elementary,High School_dropout,High_School_graduate
Diabetes_binary,1.000000,0.386039,0.269727,0.119008,0.302691,0.089251,0.117282,0.201213,-0.160012,-0.031416,...,0.204946,0.270092,0.030788,0.293554,-0.228143,0.018498,-0.159808,0.070605,0.084460,0.089238
HighBP,0.386039,1.000000,0.311303,0.102529,0.241416,0.082486,0.125699,0.210678,-0.127217,-0.020172,...,0.164488,0.235960,0.029412,0.344736,-0.182740,0.003218,-0.132206,0.043019,0.064367,0.093334
HighChol,0.269727,0.311303,1.000000,0.079531,0.140207,0.094675,0.102560,0.179972,-0.081020,-0.048002,...,0.123926,0.166127,0.009007,0.238031,-0.111971,0.009765,-0.078710,0.030140,0.032308,0.049225
CholCheck,0.119008,0.102529,0.079531,1.000000,0.052643,-0.015087,0.016615,0.038131,-0.008558,0.030746,...,0.041879,0.041385,-0.000370,0.089830,-0.006744,0.003650,-0.011538,0.011887,0.000628,0.005467
BMI,0.302691,0.241416,0.140207,0.052643,1.000000,0.013121,0.017645,0.072657,-0.170552,-0.083457,...,0.149253,0.244808,-0.004853,-0.016229,-0.124312,0.039005,-0.114813,0.015589,0.046118,0.056795
Smoker,0.089251,0.082486,0.094675,-0.015087,0.013121,1.000000,0.061009,0.121484,-0.065157,-0.078429,...,0.136578,0.123844,0.119979,0.099014,-0.086634,0.039179,-0.146539,0.000510,0.065130,0.087859
Stroke,0.117282,0.125699,0.102560,0.016615,0.017645,0.061009,1.000000,0.238585,-0.070284,0.001670,...,0.165047,0.198321,-0.007926,0.126024,-0.139339,-0.005446,-0.058051,0.026597,0.039733,0.040205
HeartDiseaseorAttack,0.201213,0.210678,0.179972,0.038131,0.072657,0.121484,0.238585,1.000000,-0.084263,-0.003457,...,0.191718,0.231392,0.086728,0.222471,-0.151817,0.009763,-0.072609,0.048165,0.028184,0.038066
PhysActivity,-0.160012,-0.127217,-0.081020,-0.008558,-0.170552,-0.065157,-0.070284,-0.084263,1.000000,0.122721,...,-0.210336,-0.264695,0.062505,-0.093880,0.186869,-0.000352,0.157677,-0.057647,-0.079536,-0.111255
Fruits,-0.031416,-0.020172,-0.048002,0.030746,-0.083457,-0.078429,0.001670,-0.003457,0.122721,1.000000,...,-0.048032,-0.042904,-0.091534,0.066394,0.045129,-0.008919,0.075690,0.011821,-0.039692,-0.057541


In [20]:
fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',  
    zmin=-1,
    zmax=1,
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title='Correlation Heatmap',
    xaxis_nticks=36,
    width=900,
    height=900
)

fig.show()

**split the data into train and test sets**

In [21]:
X = df.drop('Diabetes_binary', axis=1)
y = df['Diabetes_binary']

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=60)

**Forward Feature Selection**

In [23]:
model = SVC()
forward_feature_selection = SequentialFeatureSelector(model,
k_features=6,
forward=True,
floating=False,
verbose=2,
scoring='accuracy',
cv=5).fit(X_train, y_train)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    5.4s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  25 out of  25 | elapsed:  3.5min finished

[2025-05-01 23:46:54] Features: 1/6 -- score: 0.6892857142857143[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    8.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  24 out of  24 | elapsed:  3.5min finished

[2025-05-01 23:50:24] Features: 2/6 -- score: 0.7117857142857142[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    9.7s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  23 out of  23 | elapsed:  2.6min finished

[2025-05-01 23:53:00] Features: 3/6 -- score: 0.7285714285714285[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 

In [28]:
forward_feature_selection.k_feature_names_

('HighBP', 'BMI', 'GenHlth', 'Sex', 'Age', 'College_graduate')

In [29]:
forward_feature_selection.k_score_

0.7469047619047618

**Backward Feature Elimination**

In [31]:
backward_feature_selector = SequentialFeatureSelector(svm,
k_features=6,
forward=False,
floating=False,
verbose=2,
scoring='accuracy',
cv=5).fit(X_train, y_train)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    6.4s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  25 out of  25 | elapsed:  3.8min finished

[2025-05-02 00:39:55] Features: 24/6 -- score: 0.7478571428571428[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    6.4s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  24 out of  24 | elapsed:  2.6min finished

[2025-05-02 00:42:29] Features: 23/6 -- score: 0.7486904761904762[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    6.2s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  23 out of  23 | elapsed:  2.5min finished

[2025-05-02 00:44:59] Features: 22/6 -- score: 0.7496428571428572[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done  

In [32]:
backward_feature_selector.k_feature_names_